In [ ]:
'''
Getting the ABDU model to work in notebook

EPSG: 5070
'''
import duckdb #version 1.1.3
import geopandas as gpd #version 0.14.1
import time
from shapely import wkt
# import pandas as pd
# import pyarrow as pa
import rasterio
from rasterio import mask
from shapely.geometry import shape
from threading import Thread, current_thread

con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")
con.install_extension("azure")
con.load_extension("azure")
con.install_extension("json")
con.load_extension("json")
print(duckdb.__version__) #previously 0.9.2

# Parameters

In [ ]:
start_time = time.time()
time.ctime(start_time)

In [ ]:
aoi = '28001'
nwiurl = r"azure://abdu/nwi/**/*.parquet"
wetattrfld = 'ATTRIBUTE'
waterurl = True
politicalbndry = r'azure://abdu/uscounties.parquet'
pid_fld = 'FIPS'
hucsurl = r'azure://abdu/huc/**/*.parquet'
hucidfld = 'huc12'
crossWalk_json = 'https://giscog.blob.core.windows.net/abdu/aoiWetland.json'
nrgy_csv = r'azure://abdu/kcal.csv'
fowlDemo = r'azure://abdu/WaterfowlDemographic.parquet'
protLands = r"azure://abdu/padus/**/*.parquet",
urbanMask = "https://giscog.blob.core.windows.net/privatecogs/NLCD2016_cog.tif",
demand = r'azure://abdu/Demand9Species.parquet'
in_crs = dict() # default is 'ESPG:5070'. All will be reprojectd to crs of hucsurl. Provide param:crs pairs if source crs differs from default.


In [ ]:
params = (nwiurl, politicalbndry, hucsurl, fowlDemo, protLands, urbanMask, demand, crossWalk_json, nrgy_csv)
if any([i in '_'.join(params) for i in ('azure','giscog')]):
    con.sql("SET azure_transport_option_type = 'curl'")
    con.sql('''CREATE OR REPLACE SECRET secret0 (TYPE AZURE, ACCOUNT_NAME 'giscog')''')

In [ ]:
param_crs = {p:'EPSG:5070' for p in params[:-2]}
if len(in_crs)>0:
    for k, v in in_crs.items():
            param_crs[k] = v
param_crs 

In [ ]:
def tfrm_str(geom='geometry', in_crs='EPSG:5070', out_crs=param_crs[hucsurl]):
    """
    format str for sql query to reproject geometry of duckdb table
    """
    if not in_crs==out_crs:
        s = f"ST_Transform({geom}, '{in_crs}', '{out_crs}')"
    else:
        s = geom
    return s

In [ ]:
'''
SELECT fips geometry based on inaoifile to use as aoi for calculation.  All hucs should have center in fips.
'''
if isinstance(aoi, str):
    con.sql("""
        CREATE OR REPLACE TABLE selectedcounty AS
        SELECT {2}, geometry FROM read_parquet('{1}')
        WHERE {2} = '{0}'
    """.format(aoi, politicalbndry, pid_fld))
else:
    con.sql(f"CREATE OR REPLACE TABLE selectedcounty ({pid_fld} VARCHAR, geometry GEOMETRY)")
    # con.sql(f"INSERT INTO selectedcounty (geometry) VALUES (ST_MakeEnvelope{tuple([float(i) for i in aoi])})")
    con.sql(f"""INSERT INTO selectedcounty (geometry)
            VALUES ({tfrm_str(f'ST_MakeEnvelope{tuple([float(i) for i in aoi])}')})
             """)
    xy = con.sql(f"""SELECT {tfrm_str('ST_Centroid(geometry)', param_crs[hucsurl], 'EPSG:4326')} AS center,
        ST_X(center) AS x, ST_Y(center) AS y,        
        FROM selectedcounty
        """).to_df()[['x','y']].values[0] ## for some reason X and Y are reversed = try 'always_xy' parameter
    aoi = f"n{int(xy[0])}_e{int(xy[1])}".replace('e-','w').replace('n-','s')
    # con.sql(f"INSERT INTO selectedcounty ({pid_fld}) VALUES ('{aoi}')")
    con.sql(f"UPDATE selectedcounty SET {pid_fld} = '{aoi}'")
    print(aoi)

In [ ]:
if '**' in hucsurl:
    '''
    Read in hucs partitioned to huc2/huc4 level that have center with the aoi.  Don't clip hucs
    '''
    sql = f"""
        CREATE OR REPLACE TABLE huc12 AS
        SELECT LEFT(huc12,2) AS huc2,LEFT(huc12,4) AS huc4, huc12, areaacres, huc.geometry
        FROM (SELECT huc12, areaacres, ST_GeomFromWKB(read_parquet.geometry) AS geometry FROM read_parquet('{hucsurl}', hive_partitioning=true)
        WHERE CAST(LEFT(huc12,2) AS INTEGER)<=12) AS huc
        JOIN selectedcounty ON 
        ST_Within(ST_Centroid(huc.geometry), selectedcounty.geometry)
    """
    con.sql(sql)
    hucs = con.sql("select huc4 from huc12 GROUP BY huc4").df().values.tolist()
    hucs = sorted([item for items in hucs for item in items])
    print(hucs)
    hucidfld = 'huc4'
    wet_flds = [wetattrfld, 'huc2', hucidfld, 'huc12']
    con.execute("""
        CREATE OR REPLACE TABLE my_wetlands (
            {0} VARCHAR,
            huc2 VARCHAR,
            {1} VARCHAR,
            huc12 VARCHAR,
            geometry VARCHAR,
        )
    """.format(wetattrfld, hucidfld))
else:
    sql = f"""
        CREATE OR REPLACE TABLE huc12 AS
        SELECT * FROM read_parquet('{hucsurl}') AS huc
        JOIN selectedcounty ON 
        ST_Within(ST_Centroid(huc.geometry), selectedcounty.geometry)
    """
    con.sql(sql)
    hucs = con.sql(f"SELECT {hucidfld} from huc12").df().values.tolist()
    hucs = sorted([item for items in hucs for item in items])
    print(hucs)
    wet_flds = [wetattrfld, hucidfld]
    con.execute("""
        CREATE OR REPLACE TABLE my_wetlands (
            {0} VARCHAR,
            {1} VARCHAR,
            geometry VARCHAR,
        )
    """.format(wetattrfld, hucidfld))


# Wetland energy calculation


In [ ]:
def write_from_thread(con):
    local_con = con.cursor()
    huc = str(current_thread().name)
    if any([i in nwiurl for i in ('azure','giscog')]):
        local_con.sql('''CREATE OR REPLACE SECRET secret0 (TYPE AZURE, ACCOUNT_NAME 'giscog')''')
        local_con.sql("SET azure_transport_option_type = 'curl'")
    if '**' in nwiurl:
        sql = '''
        SELECT {2}, ST_AsWKB(ST_Intersection(ST_GeomFromWKB(wetlnd.geometry), ST_GeomFromWKB(huc12.geometry)) AS geometry
        FROM (SELECT {3}, geometry FROM read_parquet('{1}',hive_partitioning=true) 
        WHERE {4} = '{0}' AND NOT ({3} LIKE 'R%UB%' OR {3} LIKE 'R%SB%' OR {3} LIKE 'R%RB%')) AS wetlnd
        JOIN huc12 ON 
        ST_Intersects(ST_GeomFromWKB(wetlnd.geometry), ST_GeomFromWKB(huc12.geometry)))
        '''.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld)
        rows = local_con.sql(sql).fetchall()
    else:
        if waterurl:
            sql = """
            SELECT {2}, ST_AsText(ST_Intersection({5}, huc12.geometry)) AS geometry
            FROM read_parquet('{1}') wet, huc12
            WHERE huc12.{4} = '{0}'
            AND ST_Intersects({5}, huc12.geometry)
            AND NOT ST_IsEMpty(ST_Intersection({5}, huc12.geometry))
            """.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld, tfrm_str('wet.geometry', param_crs[nwiurl]))
            rows = local_con.sql(sql).fetchall()
            if isinstance(waterurl, str):
                sql = '''
                WITH wet AS
                    ({5}),
                h2o AS
                    (SELECT StandardClass, ST_Intersection(h2o.geometry, huc12.geometry) AS geometry FROM read_parquet('{1}') h2o, huc12
                    WHERE huc12.{4} = '{0}'
                    AND ST_Intersects(h2o.geometry, huc12.geometry))
                SELECT StandardClass, {4}, ST_AsText(ST_Difference(h2o.geometry, ST_GeomFromText(wet.geometry))) AS geometry
                FROM h2o, wet
                WHERE ST_Intersects(h2o.geometry, ST_GeomFromText(wet.geometry))
                AND NOT ST_IsEMpty(ST_Difference(h2o.geometry, ST_GeomFromText(wet.geometry)))
                '''.format(huc, waterurl, ', '.join(wet_flds), wetattrfld, hucidfld, sql)
                rows.extend(local_con.sql(sql).fetchall())                 
        else:
            sql = '''
                SELECT {2}, ST_AsText(ST_Intersection({5}, huc12.geometry)) AS geometry
                FROM read_parquet('{1}') wet, huc12
                WHERE huc12.{4} = '{0}'
                AND wet.{3} != 'Open Water'
                AND ST_Intersects({5}, huc12.geometry)
                AND NOT ST_IsEMpty(ST_Intersection({5}, huc12.geometry))
                '''.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld, tfrm_str('wet.geometry', param_crs[nwiurl]))
            rows = local_con.sql(sql).fetchall()
    sql = f'''INSERT INTO my_wetlands 
    VALUES ({'?, '*len(wet_flds)}?)
    '''
    result = local_con.executemany(sql, rows).fetchall()

In [ ]:
threads = []
print(hucs)
for i in range(len(hucs)):
    huc = hucs[i]
    threads.append(Thread(target = write_from_thread,
                            args = (con,),
                            name = huc))

In [ ]:
%%time
# Kick off all threads in parallel
for thread in threads:
    thread.start()

# Ensure all threads complete before printing final results
for thread in threads:
    thread.join()

# con.sql("""
#     CREATE OR REPLACE TABLE wetlands AS 
#     SELECT * FROM my_wetlands 
# """)

In [ ]:
'''
Import wetland crossclass data and assign classes to the nwi table
'''
con.sql(f"""CREATE OR REPLACE TABLE crossnwi AS
        (UNPIVOT (FROM (SELECT * FROM read_json_auto('{crossWalk_json}', maximum_object_size=100000000))) ON COLUMNS(*))""")
con.sql("""CREATE OR REPLACE TABLE crossnwi AS
        SELECT name, UNNEST(value) AS value FROM crossnwi""")
con.sql(f"""CREATE OR REPLACE TABLE wetlands AS
        SELECT name, {wet_flds[-1]}, ST_GeomFromText(geometry) AS geometry FROM my_wetlands
        LEFT JOIN crossnwi ON my_wetlands.{wetattrfld} LIKE crossnwi.value
        """)
con.sql(f"""
        CREATE OR REPLACE TABLE wetlands AS
        SELECT replace(wetlands.name, '_', '') AS name, {wet_flds[-1]}, ST_Area(geometry)*0.0001 AS ha, kcal, kcal*ha AS avalNrgy, st_buffer(geometry,0) AS geometry FROM wetlands
        LEFT JOIN read_csv_auto('{nrgy_csv}') ON replace(wetlands.name, '_', '') = read_csv_auto.habitatType
        WHERE wetlands.name IS NOT NULL
        """)
print(con.sql('SELECT count(name) FROM wetlands'))

In [ ]:
'''
Import Waterfowl demographic data to assign a fips to a specific code (breeding [4b] and non-breeding [4d])
'''
code = con.sql(f"""
SELECT code FROM read_parquet('{fowlDemo}')
JOIN selectedcounty ON 
ST_Within(ST_Centroid(selectedcounty.geometry), {tfrm_str('read_parquet.geometry')})
""").df().values.tolist()
if len(code)>0:
    code = code[0][0]
else:
    code = '4b' # Use latitude to determine?
if code is None:
    code = '4b'
print(code)

# Protected Lands

In [ ]:
'''
Read in PADUS
'''
if '**' in protLands:
    sql = """
    CREATE OR REPLACE TABLE protected AS 
    SELECT CATEGORY, huc12, huc2, huc4, ST_Intersection(ST_GeomFromWKB(huc12.geometry), ST_GeomFromWKB(prot.geometry)) as geometry
    FROM (SELECT CATEGORY, geometry FROM read_parquet('{1}', hive_partitioning=true)
    WHERE CATEGORY IN ('Fee', 'Easements', 'Other') AND huc4 IN {0}) AS prot
    JOIN huc12 ON 
    ST_Intersects(ST_GeomFromWKB(huc12.geometry), ST_GeomFromWKB(prot.geometry))
    """.format(tuple(hucs), protLands)
else:
    sql = """
    CREATE OR REPLACE TABLE protected AS
    SELECT {1}, ST_Intersection(huc12.geometry, {2}) as geometry
    FROM read_parquet('{0}') AS prot
    JOIN huc12 ON 
    ST_Intersects(huc12.geometry, {2})
    """.format(protLands, hucidfld, tfrm_str('prot.geometry'))
con.sql(sql)

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE protwetlands AS
SELECT name, wetlands.{0}, kcal, ST_Intersection(protected.geometry, wetlands.geometry) as geometry
FROM (SELECT ST_Union_Agg(geometry) as geometry from protected) as protected
JOIN wetlands ON 
ST_Intersects(wetlands.geometry, protected.geometry)
""".format(hucidfld))

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE protwetlands AS
SELECT DISTINCT geometry, name, {0}, ST_Area(geometry)*0.0001 AS ProtHabHa, kcal, kcal*ProtHabHa AS protNrgy FROM protwetlands
""".format(hucidfld))

# Urban wetlands

In [ ]:
'''
Read in NLCD clipped to hucs
'''
if 'nlcd' in urbanMask.lower() and urbanMask.endswith('.tif'):
    # Need huc12 geometry
    df = con.sql('SELECT ST_AsText(ST_geomfromwkb(geometry)) as geometry from huc12').df()
    df['geometry'] = df['geometry'].apply(wkt.loads)
    df = gpd.GeoDataFrame(df, geometry='geometry', crs=5070)
    with rasterio.open(urbanMask) as src:
        # Clip the raster to the geometry of the shapefile
        clipped_data, transform = mask.mask(src, df.geometry, crop=True)
    del df
    clipped_data[clipped_data>23]=0
    clipped_data[clipped_data<21]=0
    clipped_data[clipped_data==21]=1
    clipped_data[clipped_data==22]=1
    clipped_data[clipped_data==23]=1
    shapes = rasterio.features.shapes(clipped_data[0], transform=transform, mask=clipped_data[0] == 1)
    # Create a GeoDataFrame from the vector polygons
    gdf_vector = gpd.GeoDataFrame({'geometry': [shape(geom) for geom, value in shapes]})
    gdf_vector['geometry'] = gdf_vector.to_wkb().geometry
    sql = "CREATE OR REPLACE TABLE urban AS SELECT * EXCLUDE geometry, ST_GeomFromWKB(geometry) AS geometry FROM gdf_vector"
else:
    sql = """
    CREATE OR REPLACE TABLE urban AS
    SELECT ST_Intersection(huc12.geometry, {1}) as geometry
    FROM read_parquet('{0}') AS urb_mask
    JOIN huc12 ON ST_Intersects(huc12.geometry, {1})
    """.format(urbanMask, tfrm_str('urb_mask.geometry', param_crs[urbanMask]))
con.sql(sql)

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE urban AS 
SELECT {0}, ST_Intersection(huc12.geometry, urban.geometry) as geometry
FROM (SELECT geometry FROM urban) as urban
JOIN huc12 ON ST_Intersects(huc12.geometry, urban.geometry)
""".format(hucidfld))

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE urbanwetlands AS
SELECT name, wetlands.{0}, kcal, ST_Intersection(urban.geometry, wetlands.geometry) as geometry
FROM (SELECT geometry from urban) as urban
JOIN wetlands ON ST_Intersects(wetlands.geometry, urban.geometry)
""".format(hucidfld))

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE urbanwetlands AS
SELECT DISTINCT geometry, name, {0}, ST_Area(geometry)*0.0001 AS ha, kcal, kcal*ha AS urbanNrgy FROM urbanwetlands
""".format(hucidfld))

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE urban AS
SELECT {0}, ST_Area(geometry)*0.0001 AS urbanHa, geometry FROM urban
""".format(hucidfld))

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE unavailable AS
SELECT {0}, ST_Area(geometry)*0.0001 AS unavailHa, ST_Union_Agg(geometry) as geometry FROM
(
SELECT {0}, geometry FROM urban
UNION ALL
SELECT {0}, geometry from protected
)
group by {0}, geometry
""".format(hucidfld))

In [ ]:
#################################
#################################
#################################
#################################
#################################
#################################
#################################
#################################
#################################
#################################
'''
#################################
End of data import
Starting model process
#################################
'''
#### Prepping energy - Join energy to nwi.  Need to create the spatial kcal table first. What's the best way to do this?
# parquet is the best to read in but it's not easily editable.  Rest service would be ok but again, not great because
# reading those is difficult.  I wonder if

# Demand calculation

In [ ]:
'''
Demand

######
Need to proportion demand based on available energy.  Available energy is spatially explicit but demand is at the fips
count level.  We need to calculate total energy and demand at the huc12 scale.
To proportion demand we need to calclulate total energy by fips then calculate how much energy is in each huc12. A proportion
can then be calculated by dividing total energy within a fips by (huc12,fips) group.  Demand at the huc12 level is multiplied
by that energy proportion.
######
'''

In [ ]:
'''
Read in demand clipped by hucs
'''
con.sql("""
CREATE OR REPLACE TABLE demand AS SELECT * EXCLUDE geometry, ST_Intersection(huc12.geometry, {1}) as geometry
FROM read_parquet('{0}') dmnd
JOIN huc12 ON ST_Intersects(huc12.geometry, {1})
""".format(demand, tfrm_str('dmnd.geometry', param_crs[demand])))
con.sql(f"""CREATE OR REPLACE TABLE demand AS SELECT {hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, x80DUD, X80Demand, X80PopObj, ST_Area(geometry)*0.0001 AS ha, geometry
FROM (SELECT * FROM demand
WHERE species='All')
""")

In [ ]:
## Separate out open water kcal for comparison sake

# Energy Calculation

In [ ]:
'''Get sum of energy within huc12'''
energysum = con.sql('select sum(avalNrgy) from wetlands').df().values.tolist()[0]
print(energysum)

In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE hucdemandenergy AS 
    (SELECT name, wetlands.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, x80DUD, X80Demand, X80PopObj, kcal, 
    ST_Intersection(wetlands.geometry, demand.geometry) as geometry FROM wetlands
    JOIN demand ON ST_Intersects(wetlands.geometry, demand.geometry))""")

In [ ]:
'''
Select rows from wetland where the code is the same as pullcode
'''
con.sql("""CREATE OR REPLACE TABLE hucdemandenergy AS SELECT * from hucdemandenergy WHERE CODE = '{0}'""".format(code.upper()))

In [ ]:
'''
######
Calculate available energy (avalNrgy) of wetlands by calculating area in Hectares (HA) and multiplying by kcal.
Select only distinct rows.
Create new table habitatenergy
######
'''
#
con.sql(f"""CREATE OR REPLACE TABLE hucdemandenergy AS
        (SELECT DISTINCT name, {hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, x80DUD, X80Demand, X80PopObj, kcal, geometry, ST_Area(geometry)*0.0001 AS ha,ha*kcal AS avalNrgy 
        FROM hucdemandenergy)""")

In [ ]:
'''Total of availenergy'''
'''Get sum of energy within huc12'''
energysumfromdemand = con.sql('select sum(avalNrgy) from hucdemandenergy').df().values.tolist()[0]
print('Wetland energy: {:,.2f}'.format(energysum[0]))
print('Demand energy: {:,.2f}'.format(energysumfromdemand[0]))
dif = energysumfromdemand[0] - energysum[0]
print('Difference: {:,.0%}'.format(abs(dif/((energysumfromdemand[0] + energysum[0])/2))))


In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE rdydemand AS 
        (SELECT name, {hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, x80DUD, X80Demand, X80PopObj, kcal, avalNrgy, (hucdemandenergy.avalNrgy/{energysum[0]}) as pct, geometry
        FROM hucdemandenergy)""")

In [ ]:
# sqlcall ="""CREATE OR REPLACE TABLE rdydemand as SELECT * FROM test"""
# con.sql(sqlcall)
# con.sql("""UPDATE rdydemand SET fips ='{0}'""".format(aoi))
# print(con.sql('select fips, sum(pct) from rdydemand group by fips'))

In [ ]:
#con.sql("""describe rdydemand""")
con.sql(f"""CREATE OR REPLACE TABLE hucdemand AS (SELECT {hucidfld}, code, 
sum(pct * LTADUD) AS LTADUD,
sum(pct * LTADemand) AS LTADemand,
sum(pct * LTAPopObj) AS LTAPopObj,
sum(pct * x80DUD) AS x80DUD,
sum(pct * X80Demand) AS X80Demand,
sum(pct * X80PopObj) AS X80PopObj,
FROM rdydemand
GROUP BY {hucidfld}, code)""")
# con.sql(f"""SELECT {hucidfld}, LTADemand FROM hucdemand""")

# Summarize at huc level

In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT huc12.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj,
huc12.geometry
FROM huc12
LEFT JOIN hucdemand on hucdemand.{hucidfld} = huc12.{hucidfld}
ORDER by huc12.{hucidfld}, CODE
""")

In [ ]:
# Specified selection in a cell or two below.  Many don't need geometry at this later point.  Joining is by huc12 so selecting
# only the required columns makes the join go much faster.
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, 
sum(avalNrgy) as tothabitat_kcal, 
athuclevel.geometry
FROM athuclevel
LEFT JOIN wetlands on wetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, CODE,LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, athuclevel.geometry
ORDER by athuclevel.{hucidfld}
""")

In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, 
sum(urbanHa) as urbanHa,
athuclevel.geometry
FROM athuclevel
LEFT JOIN urban on urban.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, athuclevel.geometry
ORDER by athuclevel.{hucidfld}, CODE
""")

In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa,
sum(protNrgy) as protected_kcal,
athuclevel.geometry
FROM athuclevel
LEFT JOIN protwetlands on protwetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, CODE,LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, athuclevel.geometry
ORDER by athuclevel.{hucidfld}, CODE
""")

In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protected_kcal,
sum(ProtHabHa) as protectedhabitat_ha,
sum(protNrgy) as protected_kcal,
athuclevel.geometry
FROM athuclevel
LEFT JOIN protwetlands on protwetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, CODE,LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protected_kcal,athuclevel.geometry
ORDER by athuclevel.{hucidfld}, CODE
""")

In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal,
sum(urbanNrgy) as urbanNrgy,
athuclevel.geometry
FROM athuclevel
LEFT JOIN (SELECT {hucidfld}, urbanNrgy FROM urbanwetlands) as urbanwetlands on urbanwetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, CODE,LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal, athuclevel.geometry
""")

In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal,urbanNrgy,
sum(unavailHa) as unavailHa,
athuclevel.geometry
FROM athuclevel
LEFT JOIN (SELECT {hucidfld}, unavailHa FROM unavailable) as unavailable on unavailable.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, CODE,LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal, urbanNrgy, athuclevel.geometry
""")

In [ ]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld},
ST_Area(geometry)*0.0001 {hucidfld}_ha,
CODE, 
COALESCE(LTADUD, 0) dud_lta,
COALESCE(LTADemand,0) demand_lta_kcal, 
COALESCE(LTAPopObj,0) popobj_lta, 
COALESCE(X80DUD,0) dud_80th, 
COALESCE(X80Demand,0) demand_80th_kcal, 
COALESCE(X80PopObj,0) popobj_80th, 
COALESCE(tothabitat_kcal,0) tothabitat_kcal,
COALESCE(protected_kcal,0) protected_kcal,
COALESCE(protectedhabitat_ha,0) protectedhabitat_ha,
COALESCE(urbanHa,0) urbanHa, 
COALESCE(sum(urbanNrgy),0) urbanNrgy,
COALESCE(sum(unavailHa),0) unavailha,
COALESCE(sum(unavailHa),0) {hucidfld}_ha_unavailha,
COALESCE(tothabitat_kcal - demand_lta_kcal,0) surpdef_lta_kcal,
COALESCE(tothabitat_kcal - demand_80th_kcal,0) surpdef_80th_kcal,
athuclevel.geometry
FROM athuclevel
GROUP BY athuclevel.{hucidfld}, CODE, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, protected_kcal, protectedhabitat_ha,urbanHa, geometry
ORDER BY athuclevel.{hucidfld}, CODE
""")

In [ ]:
con.sql('''CREATE OR REPLACE TABLE athuclevel AS 
SELECT *,
CASE WHEN 
demand_lta_kcal - protected_kcal > 0
THEN
demand_lta_kcal - protected_kcal
ELSE 0
END
AS nrgprot_lta_kcal,
CASE WHEN
demand_80th_kcal - protected_kcal > 0 
THEN
demand_80th_kcal - protected_kcal
ELSE 0
END
AS nrgprot_80th_kcal
FROM athuclevel
''')

In [ ]:
'''
Calculate weighted mean
'''
con.sql(f'''
CREATE OR REPLACE TABLE wtmean AS 
SELECT huctotal.{hucidfld}, name, avalNrgname/avalNrgtot as pct, hucnametotal.kcal * pct as wtmean FROM
((SELECT {hucidfld},sum(avalNrgy) as avalNrgtot from wetlands group by {hucidfld}) huctotal
join
(SELECT {hucidfld}, name, kcal, sum(avalNrgy) as avalNrgname from wetlands group by {hucidfld}, name, kcal) hucnametotal
on hucnametotal.{hucidfld} = huctotal.{hucidfld})
''')
con.sql('''CREATE OR REPLACE TABLE wtmeanpivot AS
(select * exclude pct FROM
(pivot wtmean
    on name
    USING sum(wtmean)))
''')

In [ ]:
cols = con.sql('describe wtmeanpivot').df()['column_name'].tolist()
for cls in ('DeepwaterFresh', 'FreshMarsh', 'FreshShallowOpenWater', 'FreshwaterWoody', 'ManagedFreshMarsh', 'ManagedFreshShallowOpenWater', 'ManagedFreshwaterAquaticBed'):
    if cls not in cols:
        con.sql('''ALTER TABLE wtmeanpivot ADD COLUMN {0} DOUBLE'''.format(cls))

In [ ]:
con.sql(f'''CREATE OR REPLACE TABLE wtmeanpivot AS 
SELECT
{hucidfld},
COALESCE(DeepwaterFresh, 0) DeepwaterFresh,
COALESCE(FreshMarsh, 0) FreshMarsh, 
COALESCE(FreshShallowOpenWater, 0) FreshShallowOpenWater,
COALESCE(FreshwaterWoody, 0) FreshwaterWoody,
COALESCE(ManagedFreshMarsh, 0) ManagedFreshMarsh,
COALESCE(ManagedFreshShallowOpenWater, 0) ManagedFreshShallowOpenWater,
COALESCE(ManagedFreshwaterAquaticBed, 0) ManagedFreshwaterAquaticBed
FROM wtmeanpivot
''')
con.sql(f'''create or replace table wtmeanbyhuc as
        select {hucidfld}, 
        sum(DeepwaterFresh + FreshMarsh + FreshShallowOpenWater + FreshwaterWoody + ManagedFreshMarsh +ManagedFreshShallowOpenWater + ManagedFreshwaterAquaticBed)
        as wtmean from wtmeanpivot group by {hucidfld}''')

In [ ]:
#########
########
con.sql(f'''CREATE OR REPLACE TABLE athuclevel AS
SELECT * 
from athuclevel
left join wtmeanbyhuc on athuclevel.{hucidfld}=wtmeanbyhuc.{hucidfld}
order by athuclevel.{hucidfld}
''')
con.sql('ALTER TABLE athuclevel RENAME wtmean TO wtMean_kcal_per_ha')

In [ ]:
con.sql('''CREATE OR REPLACE TABLE athuclevel AS
SELECT *,
CASE WHEN 
surpdef_lta_kcal < 0
THEN
abs(surpdef_lta_kcal/wtMean_kcal_per_ha)
ELSE 0
END
AS restoregoal_lta_ha,

CASE WHEN 
surpdef_80th_kcal < 0
THEN
abs(surpdef_80th_kcal/wtMean_kcal_per_ha)
ELSE 0
END
AS restoregoal_80th_ha

FROM athuclevel
''')

In [ ]:
## Need to double check huc12_ha and unavailha
con.sql(f'''CREATE OR REPLACE TABLE athuclevel AS 
        select {hucidfld}, {hucidfld}_ha, CODE as code, dud_lta, demand_lta_kcal, popobj_lta, dud_80th, demand_80th_kcal, popobj_80th,
        tothabitat_kcal, protected_kcal, protectedhabitat_ha, urbanHa, urbanNrgy, unavailha, surpdef_lta_kcal, surpdef_80th_kcal,
        nrgprot_lta_kcal, nrgprot_80th_kcal, wtMean_kcal_per_ha, restoregoal_lta_ha, restoregoal_80th_ha, 
        CASE WHEN
        {hucidfld}_ha - unavailha > 0
        THEN
        {hucidfld}_ha - unavailha
        ELSE 0
        END 
        AS available_ha,
        geometry
        FROM athuclevel
        ''')

In [ ]:
con.sql('''CREATE OR REPLACE TABLE athuclevel AS
SELECT * EXCLUDE (restoregoal_lta_ha, restoregoal_80th_ha),
CASE WHEN 
restoregoal_lta_ha > available_ha
THEN
available_ha
ELSE restoregoal_lta_ha
END
AS restoregoal_lta_ha,

CASE WHEN 
restoregoal_80th_ha > available_ha
THEN
available_ha
ELSE restoregoal_80th_ha
END
AS restoregoal_80th_ha,

FROM athuclevel
''')

In [ ]:
#field='protectgoal_lta_ha', expression="(!nrgprot_lta_kcal!/!wtMean_kcal_per_ha!) if !nrgprot_lta_kcal! > 0 else 0"
#field='protectgoal_80th_ha', expression="(!nrgprot_80th_kcal!/!wtMean_kcal_per_ha!) if !nrgprot_80th_kcal! > 0 else 0"
con.sql('''CREATE OR REPLACE TABLE athuclevel AS
SELECT *,
CASE WHEN 
nrgprot_lta_kcal > 0 
THEN
nrgprot_lta_kcal/wtMean_kcal_per_ha
ELSE 0
END
AS protectgoal_lta_ha,

CASE WHEN 
nrgprot_80th_kcal > 0
THEN
nrgprot_80th_kcal/wtMean_kcal_per_ha
ELSE 0
END
AS protectgoal_80th_ha,
FROM athuclevel
''')

In [ ]:
#field='protectgoal_lta_ha', expression="!available_ha! if !protectgoal_lta_ha! > !available_ha! else !protectgoal_lta_ha!"
#field='protectgoal_80th_ha', expression="!available_ha! if !protectgoal_80th_ha! > !available_ha! else !protectgoal_80th_ha!"
con.sql('''CREATE OR REPLACE TABLE athuclevel AS
SELECT * EXCLUDE (protectgoal_lta_ha, protectgoal_80th_ha),
CASE WHEN 
protectgoal_lta_ha > available_ha
THEN
available_ha
ELSE protectgoal_lta_ha
END
AS  protectgoal_lta_ha,

CASE WHEN 
protectgoal_80th_ha > available_ha
THEN
available_ha
ELSE protectgoal_80th_ha
END
AS protectgoal_80th_ha,
FROM athuclevel
''')

In [ ]:
'''
Protected wetlands, urban wetlands, and wetland energy all calculated by huc12.  Need to calculate total urban outside of
wetland energy

Calculations:
    Energy supply
        Total habitat energy within huc - THabNrg
        Total habitat hectares within huc - THabHA

    Energy demand
        LTA and X80 DUD by huc - TLTADUD anc X80DUD
        LTA and X80 Demand by huc - TLTADemand and X80Demand
        LTA and X80 Population objective by huc - LTAPopObj and X80PopObj
        
    Protected lands
        Total protected hectares by huc - ProtHA

    Protected habitat hectares and energy
        Total protected hectares - ProtHabHA
        Total protected energy - ProtHabNrg

    Weighted mean and calculations based off of it
        Weighted mean kcal/ha with weight being Total habitat energy
        Energy Protection needed - NrgProtRq
        Restoration HA based off of weighted mean - RstorHA
        Protection HA based off weighted mean - RstorProtHA  

'''
#################################
#################################
#################################


# Save output to file

In [ ]:
con.sql("""COPY (SELECT * FROM athuclevel) TO './output/{0}.parquet' (FORMAT PARQUET)""".format(aoi))
print('Done in {0:.1f} seconds'.format(time.time() - start_time))